# Melkior_NN
Neural Network classifier pipeline — training, evaluation (AUC & AMS significance) on the HiggsML dataset.

In [14]:
COLAB = "google.colab" in str(get_ipython())

if COLAB:
    !rm -rf Higgs_collaborations
    !git clone https://github.com/Higgs-A/Alpaca-new.git
    %cd Alpaca-new
    %cd Higgs_collaborations

    !git checkout dev

Cloning into 'Alpaca-new'...
remote: Enumerating objects: 1240, done.
remote: Counting objects: 100% (229/229), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 1240 (delta 120), reused 125 (delta 62), pack-reused 1011 (from 1)
Receiving objects: 100% (1240/1240), 43.28 MiB | 20.91 MiB/s, done.
Resolving deltas: 100% (605/605), done.
/content/Alpaca-new/Higgs_collaborations/Alpaca-new
/content/Alpaca-new/Higgs_collaborations/Alpaca-new/Higgs_collaborations
Branch 'dev' set up to track remote branch 'dev' from 'origin'.
Switched to a new branch 'dev'


In [15]:
!pip install HiggsML

  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)


In [16]:
import os
import joblib
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
tf.config.run_functions_eagerly(False)

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler


class NeuralNetwork:

    def __init__(self, n_features=None):
        self.model  = None
        self.scaler = StandardScaler()

        self._predictions  = None
        self._test_labels  = None
        self._test_weights = None

        if n_features is not None:
            self._initialize_model(n_features)

    def _initialize_model(self, n_features):
        """Initialize the model architecture."""
        self.model = Sequential([
            Dense(256, input_dim=n_features, activation="relu"),
            BatchNormalization(), Dropout(0.3),

            Dense(256, activation="relu"),
            BatchNormalization(), Dropout(0.3),

            Dense(128, activation="relu"),
            BatchNormalization(), Dropout(0.3),

            Dense(128, activation="relu"),
            BatchNormalization(), Dropout(0.3),

            Dense(64, activation="relu"),
            BatchNormalization(), Dropout(0.3),

            Dense(1, activation="sigmoid"),
        ])

        self.model.compile(
            optimizer=Adam(learning_rate=1e-3),
            loss="binary_crossentropy",
            # AUC metric here is unweighted — same scale as val_auc.
            # Training loss is still computed with sample_weight (balanced),
            # so the optimiser still sees the rebalanced signal.
            metrics=[AUC(name="auc")],
        )

    def fit(self, train_data, y_train, weights_train=None):
        """Train the model."""
        if self.model is None:
            raise ValueError(
                "Model is not initialized. Ensure `_initialize_model` is called or load a saved model."
            )

        X_train = self.scaler.fit_transform(train_data)

        callbacks = [
            EarlyStopping(
                monitor="val_auc", mode="max",
                patience=10, min_delta=3e-5,
                restore_best_weights=True, verbose=1,
            ),
            ReduceLROnPlateau(
                monitor="val_auc", mode="max",
                factor=0.5, patience=5, min_lr=1e-6, verbose=1,
            ),
        ]

        self.history = self.model.fit(
            X_train, y_train,
            sample_weight=weights_train,
            epochs=100,
            batch_size=512,
            validation_split=0.2,
            callbacks=callbacks,
            verbose=2,
        )
        # Recompute unweighted train AUC after training so learning curve
        # shows train and val on the same unweighted scale.
        from sklearn.metrics import roc_auc_score as _ras
        train_preds = self.model.predict(X_train, verbose=0).ravel()
        self.auc_train_unweighted = float(_ras(y_train, train_preds))

    def predict(self, test_data, labels=None, weights=None):
        self._predictions = self.model.predict(
            self.scaler.transform(test_data), verbose=0
        ).ravel()

        if labels  is not None: self._test_labels  = np.asarray(labels)
        if weights is not None: self._test_weights = np.asarray(weights)

        return self._predictions

    def significance(self, test_labels=None, test_weights=None):
        if test_labels  is not None: self._test_labels  = np.asarray(test_labels)
        if test_weights is not None: self._test_weights = np.asarray(test_weights)

        if self._predictions is None:
            raise ValueError("No predictions found. Call predict() first.")
        if self._test_labels is None:
            raise ValueError(
                "True labels for test data are not available. Please provide them when calling predict()."
            )

        # Official AMS formula from the HiggsML starting kit:
        #   AMS(s, b) = sqrt(2 * ((s + b + bReg) * log(1 + s/(b+bReg)) - s))
        # bReg = 10 stabilises the formula at low b.
        # Weights are already physically normalised at load time (S=1015, B=1050370),
        # so no further wFactor scaling is needed here.
        B_REG = 10.0

        def __ams(s, b):
            s = np.asarray(s, float)
            b = np.asarray(b, float)
            val = np.sqrt(2 * ((s + b + B_REG) * np.log(1 + s / (b + B_REG)) - s))
            val = np.where(s < 0, np.nan, val)
            if np.isscalar(s):
                return float(val)
            return val

        def __significance_vscore(y_true, y_score, sample_weight):
            sample_weight = np.asarray(sample_weight)
            bins = np.linspace(0, 1.0, 101)
            s_hist, _ = np.histogram(
                y_score[y_true == 1], bins=bins, weights=sample_weight[y_true == 1]
            )
            b_hist, _ = np.histogram(
                y_score[y_true == 0], bins=bins, weights=sample_weight[y_true == 0]
            )
            s_cumul = np.cumsum(s_hist[::-1])[::-1]
            b_cumul = np.cumsum(b_hist[::-1])[::-1]
            return __ams(s_cumul, b_cumul)

        ams_curve = __significance_vscore(
            y_true=self._test_labels,
            y_score=self._predictions,
            sample_weight=self._test_weights,
        )

        plt.plot(np.linspace(0, 1.0, 100), ams_curve, label="AMS Significance")
        plt.xlabel("Score threshold")
        plt.ylabel("AMS")
        return float(np.nanmax(ams_curve))

    def plot_learning_curves(self, weighted_test_auc=None, unweighted_test_auc=None):
        if not hasattr(self, "history"):
            raise ValueError("Model must be trained before plotting learning curves.")
        fig, ax1 = plt.subplots(figsize=(10, 5))
        ax2 = ax1.twinx()

        # val_auc from Keras is unweighted (Keras ignores sample_weight for val metrics).
        # history["auc"] is weighted (balanced weights used during training).
        # To make train vs val comparable we show the unweighted train AUC as a dot
        # at the final epoch, alongside the val_auc curve.
        l2, = ax2.plot(self.history.history["loss"],    color="tab:orange", label="Loss (train)")
        lines = [l2]
        if "val_auc" in self.history.history:
            l3, = ax1.plot(self.history.history["val_auc"], color="tab:blue", linestyle="--",
                           label="AUC (internal val, unweighted)")
            lines.append(l3)
        if "val_loss" in self.history.history:
            l4, = ax2.plot(self.history.history["val_loss"], color="tab:orange", linestyle="--",
                           label="Loss (internal val)")
            lines.append(l4)
        if hasattr(self, "auc_train_unweighted"):
            l1 = ax1.axhline(self.auc_train_unweighted, color="tab:blue", linestyle="-", linewidth=1.2,
                             label=f"AUC (train, unweighted) = {self.auc_train_unweighted:.4f}")
            lines.append(l1)
        if unweighted_test_auc is not None:
            l5 = ax1.axhline(unweighted_test_auc, color="tab:green", linestyle=":", linewidth=1.5,
                             label=f"AUC (test, unweighted) = {unweighted_test_auc:.4f}")
            lines.append(l5)
        if weighted_test_auc is not None:
            l6 = ax1.axhline(weighted_test_auc, color="tab:red", linestyle=":", linewidth=1.5,
                             label=f"AUC (test, weighted) = {weighted_test_auc:.4f}")
            lines.append(l6)

        ax1.set_xlabel("Epochs")
        ax1.set_ylabel("AUC",  color="tab:blue")
        ax2.set_ylabel("Loss", color="tab:orange")
        ax1.tick_params(axis="y", labelcolor="tab:blue")
        ax2.tick_params(axis="y", labelcolor="tab:orange")
        ax1.legend(lines, [l.get_label() for l in lines])
        ax1.grid(True)
        plt.title("Learning Curves — all AUC values unweighted for comparability")
        plt.tight_layout()
        plt.show()

    def plot_score_distribution(self, X_test, y_test):
        y_pred = self.predict(X_test)

        signal_scores = y_pred[y_test == 1]
        bkg_scores    = y_pred[y_test == 0]

        plt.figure(figsize=(8, 6))
        plt.hist(signal_scores, bins=50, alpha=0.5, label='Signal',     color='blue', density=True)
        plt.hist(bkg_scores,    bins=50, alpha=0.5, label='Background', color='red',  density=True)
        plt.title('Score Distribution (Signal vs Background)')
        plt.xlabel('Prediction Score')
        plt.ylabel('Density')
        plt.legend()
        plt.grid(True)
        plt.show()

    def save_model(self, path):
        """Save the trained model and scaler to the specified path."""
        os.makedirs(path, exist_ok=True)
        self.model.save(os.path.join(path, "model.keras"))
        joblib.dump(self.scaler, os.path.join(path, "scaler.pkl"))
        print(f"Model saved to {path}")

    def load_model(self, path):
        """Load the trained model and scaler from the specified path."""
        self.model  = load_model(os.path.join(path, "model.keras"))
        self.scaler = joblib.load(os.path.join(path, "scaler.pkl"))
        print(f"Model loaded from {path}")

### Load Physical Data
We load the dataset using the same split as the starting kit:
- **Test set**: first `test_size` rows, split by physics process (ztautau, diboson, ttbar, htautau).
- **Train set**: remaining rows, with balanced weights.

In [17]:
import pandas as pd
from HiggsML.datasets import download_dataset

dataset_name = "blackSwan_data"
print(f"Loading dataset: {dataset_name}")
data = download_dataset(dataset_name)

# --- Load train set (random sampling, intentionally non-deterministic) ---
data.load_train_set()
df_train = data.get_train_set()

feature_cols = [c for c in df_train.columns if c not in ("labels", "weights", "detailed_labels")]

X_train       = df_train[feature_cols].values
y_train       = df_train["labels"].values
weights_train = df_train["weights"].values

# --- Load test set (deterministic: always the same first test_size rows) ---
data.load_test_set()
test_set = data.get_test_set()   # dict: {"ztautau", "diboson", "ttbar", "htautau"}

signal_df = test_set["htautau"].copy()
signal_df["labels"] = 1

bkg_frames = []
for proc in ("ztautau", "diboson", "ttbar"):
    df = test_set[proc].copy()
    df["labels"] = 0
    bkg_frames.append(df)

df_test = pd.concat([signal_df] + bkg_frames, ignore_index=True)

X_test       = df_test[feature_cols].values
y_test       = df_test["labels"].values
weights_test = df_test["weights"].values

# --- Rescale weights to match known physical LHC event yields ---
# From the dataset documentation (https://zenodo.org/records/15131565):
#   Signal (htautau) : 1015 expected LHC events
#   Background total : 1,002,395 + 3,783 + 44,192 = 1,050,370 expected LHC events
# The HiggsML library rescales weights per-batch independently, which distorts
# the absolute scale. We undo this by rescaling each class to its known physical sum.
SUM_W_SIG = 1_015.0
SUM_W_BKG = 1_050_370.0

def rescale_weights(weights, labels, sum_sig, sum_bkg):
    w = weights.copy().astype(float)
    sig_mask = labels == 1
    bkg_mask = labels == 0
    if w[sig_mask].sum() > 0:
        w[sig_mask] *= sum_sig / w[sig_mask].sum()
    if w[bkg_mask].sum() > 0:
        w[bkg_mask] *= sum_bkg / w[bkg_mask].sum()
    return w

weights_train = rescale_weights(weights_train, y_train, SUM_W_SIG, SUM_W_BKG)
weights_test  = rescale_weights(weights_test,  y_test,  SUM_W_SIG, SUM_W_BKG)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape:  {X_test.shape}")
print(f"Test sum W signal:   {weights_test[y_test==1].sum():.1f}  (expected ~1015)")
print(f"Test sum W bkg:      {weights_test[y_test==0].sum():.1f}  (expected ~1050370)")

# --- Rebalance training weights ---
# Signal weights are ~75x smaller than background weights.
# Without rebalancing the model learns to predict background for everything.
# We scale signal weights up so total signal weight == total background weight.
# The original weights are kept unchanged for AUC / AMS evaluation.
w_sig = weights_train[y_train == 1].sum()
w_bkg = weights_train[y_train == 0].sum()
weights_train_balanced = weights_train.copy()
weights_train_balanced[y_train == 1] *= w_bkg / w_sig
print(f"Weight scale applied to signal: {w_bkg / w_sig:.1f}x")


Loading dataset: blackSwan_data
Training data shape: (1400000, 28)
Testing data shape:  (600000, 28)
Test sum W signal:   1015.0  (expected ~1015)
Test sum W bkg:      1050370.0  (expected ~1050370)
Weight scale applied to signal: 1034.8x


### Train the Model

In [ ]:
nn = NeuralNetwork(n_features=X_train.shape[1])
print("Starting training...")

# --- Rebalance training weights ---
# Signal weights are ~75x smaller than background weights.
# Without rebalancing the model learns to predict background for everything.
# We scale signal weights up so total signal weight == total background weight.
# The original weights are kept unchanged for AUC / AMS evaluation.
w_sig = weights_train[y_train == 1].sum()
w_bkg = weights_train[y_train == 0].sum()
weights_train_balanced = weights_train.copy()
weights_train_balanced[y_train == 1] *= w_bkg / w_sig
print(f"Weight scale applied to signal: {w_bkg / w_sig:.1f}x")

nn.fit(X_train, y_train, weights_train=weights_train_balanced)
print("Training complete")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Starting training...
Weight scale applied to signal: 1034.8x
Epoch 1/100
2188/2188 - 25s - 12ms/step - auc: 0.8552 - loss: 0.7090 - val_auc: 0.8750 - val_loss: 0.6586 - learning_rate: 0.0010
Epoch 2/100
2188/2188 - 11s - 5ms/step - auc: 0.8701 - loss: 0.6696 - val_auc: 0.8776 - val_loss: 0.6503 - learning_rate: 0.0010
Epoch 3/100
2188/2188 - 10s - 4ms/step - auc: 0.8725 - loss: 0.6635 - val_auc: 0.8777 - val_loss: 0.6501 - learning_rate: 0.0010
Epoch 4/100
2188/2188 - 10s - 5ms/step - auc: 0.8740 - loss: 0.6596 - val_auc: 0.8797 - val_loss: 0.6449 - learning_rate: 0.0010
Epoch 5/100
2188/2188 - 10s - 5ms/step - auc: 0.8749 - loss: 0.6575 - val_auc: 0.8792 - val_loss: 0.6457 - learning_rate: 0.0010
Epoch 6/100
2188/2188 - 10s - 5ms/step - auc: 0.8756 - loss: 0.6558 - val_auc: 0.8805 - val_loss: 0.6427 - learning_rate: 0.0010
Epoch 7/100
2188/2188 - 10s - 5ms/step - auc: 0.8761 - loss: 0.6543 - val_auc: 0.8806 - val_loss: 0.6425 - learning_rate: 0.0010
Epoch 8/100
2188/2188 - 10s - 5ms/s

### Test the Model and Evaluate Metrics (AUC & Significance)

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

# Predict on the deterministic test set (model has never seen this data)
predictions = nn.predict(X_test, labels=y_test, weights=weights_test)

# --- AUC (weighted and unweighted) ---
auc_weighted   = roc_auc_score(y_test, predictions, sample_weight=weights_test)
auc_unweighted = roc_auc_score(y_test, predictions)
print(f"ROC AUC (weighted)  : {auc_weighted:.4f}  — accounts for physical event yields")
print(f"ROC AUC (unweighted): {auc_unweighted:.4f}  — raw ranking ability, comparable with other teams")

# --- ROC curves side by side ---
fpr_w,  tpr_w,  _ = roc_curve(y_test, predictions, sample_weight=weights_test)
fpr_u,  tpr_u,  _ = roc_curve(y_test, predictions)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(fpr_w, tpr_w, color='blue', label=f'Weighted AUC = {auc_weighted:.4f}')
ax1.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve (weighted)')
ax1.legend(loc='lower right')
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.plot(fpr_u, tpr_u, color='orange', label=f'Unweighted AUC = {auc_unweighted:.4f}')
ax2.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve (unweighted)')
ax2.legend(loc='lower right')
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# --- Significance ---
plt.figure(figsize=(8, 6))
max_significance = nn.significance()
plt.title('Significance vs. Threshold')
plt.show()
print(f"Max Significance: {max_significance:.4f}")

# --- Learning curves (weighted test AUC shown as reference line) ---
print("\nPlotting learning curves...")
nn.plot_learning_curves(weighted_test_auc=auc_weighted, unweighted_test_auc=auc_unweighted)

# --- Score distribution (test set) ---
print("\nPlotting score distribution...")
nn.plot_score_distribution(X_test, y_test)


### Save and Load Models

**Single model** (e.g. the one trained in the first section): call `nn.save_model(path)`.

**Ensemble**: each model is saved to its own subdirectory `models/model_i/`. To reload the full ensemble later, re-run the loop below.

To save only the best model from the ensemble: `models[best_idx].save_model("models/best")`.

In [ ]:
nn.save_model("models/single")

# To reload it:
# nn_loaded = NeuralNetwork()
# nn_loaded.load_model("models/single")


### Download Models to Google Drive
Zip any saved model folder and copy it to Google Drive so you can download it from drive.google.com.

In [ ]:
# import shutil
# from google.colab import drive

# # Mount Google Drive (will ask for authorisation once)
# drive.mount("/content/drive")

# # Zip whichever model you want to download, then copy to Drive.
# # Change the paths below as needed.

# # --- Best model ---
# shutil.make_archive("best_model", "zip", "models/best")
# shutil.copy("best_model.zip", "/content/drive/MyDrive/best_model.zip")
# print("best_model.zip saved to Google Drive")

# # --- Full ensemble (optional) ---
# # shutil.make_archive("ensemble", "zip", "models/ensemble")
# # shutil.copy("ensemble.zip", "/content/drive/MyDrive/ensemble.zip")
# # print("ensemble.zip saved to Google Drive")

# print("\nGo to drive.google.com to download the zip file(s).")